In [ ]:
import logging
from typing import Iterable


__all__ = ["MegatronPPOActor"]

logger = logging.getLogger(__file__)
logger.setLevel(os.getenv("VERL_LOGGING_LEVEL", "WARN"))



class MegatronPPOActor(BasePPOActor):
    
    @GPUMemoryLogger(role="megatron actor", logger=logger)
    def update_policy(self, dataloader: Iterable[DataProto]) -> dict:
        # """Update the policy with an iterator of DataProto

        # Args:
        #     dataloader (Iterable[DataProto]): an iterator over the DataProto that returns by ``make_minibatch_iterator``
        #         The keys of each data batch is described in the make_minibatch_iterator.

        # Returns:
        #     Dict: a dictionary containing the statistics. Note that the statistics are only valid in the last pp stage
        #     and users have to combine the output in each dp rank manually.

        # """
        metrics = {}
        self.prof.start()
        for data in dataloader:
            data.to(get_device_id())
            self.actor_optimizer.zero_grad()
            # use use_contiguous_buffers_in_local_ddp and no overlap_dp_param_comm
            for chunk in self.actor_module:
                # if use distributed optimizer, zero grad buffer will be handled by optimizer
                chunk.zero_grad_buffer()

            calculate_entropy = self.config.entropy_coeff != 0
            if data.meta_info.get("micro_batch_size", None) is not None:
                micro_batch_size = data.meta_info["micro_batch_size"]
            else:
                micro_batch_size = self.config.ppo_micro_batch_size_per_gpu
            max_token_len = None
            if self.config.use_dynamic_bsz:
                max_token_len = self.config.ppo_max_token_len_per_gpu * self.config.megatron.context_parallel_size
            
            '''
            这是计算量最大的一步，它会将一个大的批次拆分成多个微批次，
            依次送入模型进行前向推理（计算对数概率、熵等）和反向传播（计算梯度）。
            
            计算出的各项指标（如 Loss、Entropy）会被收集到 metrics 字典中，用于后续的监控和分析
            '''
            metric_micro_batch = self.forward_backward_batch(
                data,
                calculate_entropy=calculate_entropy,
                use_dynamic_bsz=self.config.use_dynamic_bsz,
                micro_batch_size=micro_batch_size,
                max_token_len=max_token_len,
                mini_batch_size=self.config.ppo_mini_batch_size,
            )
            metric_micro_batch = metric_micro_batch["output"]
            for metric in metric_micro_batch:
                # Note that o[0] is metrics, o[1] is entropy, o[2] is response_mask
                append_to_dict(metrics, metric[0])  # append the metric from this micro-batch to global metrics.

            update_successful, grad_norm, num_zeros_in_grad = self.actor_optimizer.step()
            data = {"actor/grad_norm": grad_norm}
            append_to_dict(metrics, data)

            if update_successful:
                # allgather already execute in optimizer.step in new megatron
                pass
            else:
                raise NotImplementedError
            self.prof.step()
        # add empty cache after each compute
        self.prof.stop_and_save()
        self.prof.stop_trace()
        get_torch_device().empty_cache()
        return metrics


这段代码是一个基于 Megatron 框架的大模型强化学习（PPO）策略更新的核心方法。其中，@GPUMemoryLogger(role="megatron actor", logger=logger) 是一个自定义的 Python 装饰器，而 update_policy 则是实际执行模型梯度计算与参数更新的主函数。

下面为你详细拆解这两部分的含义：

## 1. 装饰器 @GPUMemoryLogger 的作用
@GPUMemoryLogger(role="megatron actor", logger=logger) 并不是 PyTorch 或 Megatron 官方自带的原生组件，而是该项目（推测为 veRL 或类似的强化学习框架）内部封装的一个性能监控工具。
- 功能推测：<font color='red'>它的核心作用是在 update_policy 方法执行的前后，自动记录当前 GPU 的显存占用情况。</font>
- 参数解析：
    - role="megatron actor"：给当前的监控任务打上标签。在强化学习（如 PPO）中，通常有 Actor（策略网络）、Critic（价值网络）、Reference（参考网络）等多个角色。打上标签后，打印出来的日志就能清晰地知道是哪一部分吃掉了显存。
    - logger=logger：指定将显存日志输出到哪个日志记录器中。
- 实际意义：在大模型训练中，显存（VRAM）是最稀缺的资源。<font color='red'>通过这个装饰器，开发者可以精准定位到“策略更新”这一步到底消耗了多少显存，从而方便进行显存优化（比如开启参数卸载、调整微批次大小等）。</font>

## 2. update_policy 方法的核心流程
这个方法实现了 PPO 算法中“更新 Actor 策略网络”的完整闭环，主要包含以下几个关键阶段：

### ① 初始化与梯度清零
- self.prof.start()：启动性能分析器（Profiler），用于记录整个更新过程的耗时。
- self.actor_optimizer.zero_grad()：清空优化器中残留的旧梯度。
- chunk.zero_grad_buffer()：<font color='red'>针对 Megatron 分布式训练的特性，清空分布式通信缓冲区中的梯度。</font>

### ② 准备微批次（Micro-batch）配置
- 代码会动态决定每个微批次的大小（micro_batch_size）。如果开启了动态批次大小（use_dynamic_bsz），还会根据配置计算当前 GPU 允许的最大 Token 长度，以防止显存溢出（OOM）。

### ③ 前向与反向传播（Forward & Backward）
- <font color='red'>调用 self.forward_backward_batch(...)。这是计算量最大的一步，它会将一个大的批次拆分成多个微批次，依次送入模型进行前向推理（计算对数概率、熵等）和反向传播（计算梯度）。</font>
- <font color='green'>计算出的各项指标（如 Loss、Entropy）会被收集到 metrics 字典中，用于后续的监控和分析。</font>

### ④ 优化器步进（Optimizer Step）
- update_successful, grad_norm, num_zeros_in_grad = self.actor_optimizer.step()：<font color='red'>这是参数实际更新的时刻。优化器会根据计算出的梯度，结合 Adam 等算法的规则，去更新模型的 FP32 主权重副本。</font>
- 同时，<font color='green'>它会返回梯度的范数（grad_norm），这是一个非常重要的指标，用来判断训练过程中是否出现了梯度爆炸或梯度消失。</font>

### ⑤ 异常处理与收尾
- 如果更新成功（update_successful），则正常结束；如果失败（例如梯度出现了 NaN 非数值），代码会触发异常（目前代码中是 raise NotImplementedError，实际项目中通常会记录警告并跳过该批次）。
- self.prof.stop_and_save()：停止性能分析并保存报告。
- get_torch_device().empty_cache()：主动清理 PyTorch 的显存缓存，释放碎片化显存，为下一轮训练腾出空间。

总结来说，这段代码是一个高度工程化的大模型 PPO 训练更新单元。装饰器负责在后台默默监控显存健康度，而函数内部则严谨地执行了“清零梯度 -> 计算新梯度 -> 更新参数 -> 释放资源”的标准训练流水线。

# forward_backward_batch 前向推理和反向传播
- <font color='red'>调用 self.forward_backward_batch(...)。这是计算量最大的一步，它会将一个大的批次拆分成多个微批次，依次送入模型进行前向推理（计算对数概率、熵等）和反向传播（计算梯度）。</font>



In [ ]:
    def forward_backward_batch(
        self,
        data: DataProto,
        forward_only=False,
        post_process_fn=None,
        calculate_entropy=False,
        use_dynamic_bsz=False,
        micro_batch_size=None,
        max_token_len=None,
        mini_batch_size=None,
    ):
        """
        We assume:
        - The model takes input: (input_ids, attention_mask, position_ids). No rmpad for the input
        - The communication shape is (total_nnz_pad_to_sp // tp_size, 1, hidden_size) if sequence parallel is enabled
        """
        # broadcast from last pp rank to all other pp ranks
        # TODO: actually, we just need to control the sampling order.
        mini_batch = data
        broadcast_dict_tensor(
            mini_batch.batch,
            src=mpu.get_pipeline_model_parallel_last_rank(),
            group=mpu.get_pipeline_model_parallel_group(),
        )
        
        # split into micro-batches
        mini_batch.batch["attention_mask"] = mini_batch.batch["attention_mask"].to(bool)
        self.has_multi_modal_inputs = "multi_modal_inputs" in mini_batch.non_tensor_batch.keys()
        if self.has_multi_modal_inputs:
            mini_batch.batch["multi_modal_inputs"] = mini_batch.non_tensor_batch["multi_modal_inputs"]
            mini_batch.batch["multi_modal_inputs_idx"] = torch.Tensor(
                list(range(len(mini_batch.non_tensor_batch["multi_modal_inputs"])))
            ).to(torch.int64)

        if mini_batch.batch["position_ids"].dim() == 3:  # qwen2vl mrope [bs, 3, seq_len]
            mini_batch.batch["position_ids"] = mini_batch.batch["position_ids"][
                :, 0
            ]  # mcore patch recompute qwen2vl's pos ids during forward

        indices = None
        if use_dynamic_bsz:
            assert max_token_len is not None, "max_token_len must be set when use_dynamic_bsz is True"
            vpp_size = mpu.get_virtual_pipeline_model_parallel_world_size()
            if vpp_size is not None and vpp_size > 1:
                microbatch_group_size_per_vp_stage = self.tf_config.microbatch_group_size_per_vp_stage
                micro_batches, indices = rearrange_micro_batches(
                    batch=mini_batch.batch,
                    num_batches_divided_by=microbatch_group_size_per_vp_stage,
                    max_token_len=max_token_len,
                )
                assert len(micro_batches) % self.tf_config.microbatch_group_size_per_vp_stage == 0, (
                    f"micro_batches {micro_batches} must be divisible by microbatch_group_size_per_vp_stage "
                    f"{microbatch_group_size_per_vp_stage} for megatron backend"
                )
            else:
                micro_batches, indices = rearrange_micro_batches(batch=mini_batch.batch, max_token_len=max_token_len)
            total_seqlen = max_token_len
        else:
            assert micro_batch_size is not None, (
                "micro_batch_size is needed to be passed in when not using dynamic batch size"
            )
            micro_batches = mini_batch.batch.split(micro_batch_size)
            seq_len = micro_batches[0]["input_ids"].shape[1]
            total_seqlen = micro_batch_size * seq_len
        # compute input shapes for pp stages
        n_micro_batch = len(micro_batches)

        '''
        获取调度器：
        '''
        forward_backward_func = get_forward_backward_func()

        def loss_func(output, data, meta_info):
            # For memory efficiency
            # We move calculation of entropy to compute_log_probs, forward_only == True
            device = output["log_probs"].device
            metrics = {}
            if forward_only:
                if post_process_fn is None:
                    pass
                    # metrics["logits"] = output
                else:
                    stats = post_process_fn(output, data)
                    metrics.update(stats)
                if not calculate_entropy:
                    return torch.tensor(1.0, device=device), metrics

            responses = data["responses"]
            response_length = responses.size(1)
            response_mask = data["response_mask"].to(bool)
            loss_agg_mode = self.config.loss_agg_mode

            # compute policy loss
            log_prob = output["log_probs"][:, -response_length - 1 : -1].contiguous()
            ret_entropy = None
            stats = {}
            if not forward_only:
                old_log_prob = data["old_log_probs"]
                advantages = data["advantages"]

                entropy_coeff = self.config.entropy_coeff
                loss_agg_mode = self.config.loss_agg_mode

                loss_mode = self.config.policy_loss.get("loss_mode", "vanilla")
                
                policy_loss_fn = get_policy_loss_fn(loss_mode)
                # pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower = policy_loss_fn(
                #     old_log_prob=old_log_prob,
                #     log_prob=log_prob,
                #     advantages=advantages,
                #     response_mask=response_mask,
                #     loss_agg_mode=loss_agg_mode,
                #     config=self.config,
                # )
                if loss_mode == "future_kl":
                    pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower, influence_weights_mean, influence_weights_min, influence_weights_max,total_clip_frac,clip_frac_upper, \
                    clip_frac_lower,influence_weights_mean_raw, raw_influence_weights_min, raw_influence_weights_max, neg_ratio_2_3, neg_ratio_3_4, neg_ratio_4_10,\
                    neg_is_max, neg_is_p995,neg_is_p999,neg_is_p75, pos_is_max, pos_is_median, pos_is_p75, pos_is_p995, pos_is_p999,pos_is_p25, \
                    pos_is_min, pos_mini_frac, negative_approx_kl  = policy_loss_fn(
                    old_log_prob=old_log_prob,
                    log_prob=log_prob,
                    advantages=advantages,
                    response_mask=response_mask,
                    loss_agg_mode=loss_agg_mode,
                    config=self.config,
                        )

                    stats.update(
                        {
                            "actor/pg_clipfrac": pg_clipfrac.detach().item(),
                            "actor/ppo_kl": ppo_kl.detach().item(),
                            "actor/pg_clipfrac_lower": pg_clipfrac_lower.detach().item(),
                            "actor/influence_weights_mean": influence_weights_mean.detach().item(),
                            "actor/influence_weights_min":influence_weights_min.detach().item(),
                            "actor/influence_weights_max": influence_weights_max.detach().item(),
                            "actor/IW_overall_clip_ratio": total_clip_frac.detach().item(),
                            "actor/IW_upper_clip_ratio": clip_frac_upper.detach().item(),
                            "actor/IW_lower_clip_ratio": clip_frac_lower.detach().item(),
                            # raw influence weight (before clip)
                            "actor/influence_weights_mean_raw":influence_weights_mean_raw.detach().item(),
                            "actor/raw_influence_weights_min": raw_influence_weights_min.detach().item(),
                            "actor/raw_influence_weights_max":  raw_influence_weights_max.detach().item(),
                            # negative sample importance sampling ratio info
                            "actor/neg_ratio_2_3": neg_ratio_2_3.detach().item(),
                            "actor/neg_ratio_3_4": neg_ratio_3_4.detach().item(),
                            "actor/neg_ratio_4_10": neg_ratio_4_10.detach().item(),
                            # negative sample IS ratio basic stats
                            "actor/neg_is_max": neg_is_max.detach().item(),
                            "actor/neg_is_p995": neg_is_p995.detach().item(),
                            "actor/neg_is_p999": neg_is_p999.detach().item(),
                            "actor/neg_is_p75": neg_is_p75.detach().item(),
                            # postive sample IS ratio basic stats
                            "actor/pos_is_max":pos_is_max.detach().item(),
                            "actor/pos_is_median":pos_is_median.detach().item(),
                            "actor/pos_is_p75": pos_is_p75.detach().item(),
                            "actor/pos_is_p995": pos_is_p995.detach().item(),
                            "actor/pos_is_p999": pos_is_p999.detach().item(),
                            "actor/pos_is_p25": pos_is_p25.detach().item(),
                            "actor/pos_is_min": pos_is_min.detach().item(),
                            "actor/pos_mini_frac": pos_mini_frac.detach().item()

                        }   
                    )
                    stats["actor/global_log_buffer"] = {
                        "negative_approx_kl": negative_approx_kl.detach().cpu(),
                        "responses": responses.detach().cpu()
                    }
                else:
                    pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower = policy_loss_fn(
                    old_log_prob=old_log_prob,
                    log_prob=log_prob,
                    advantages=advantages,
                    response_mask=response_mask,
                    loss_agg_mode=loss_agg_mode,
                    config=self.config,
                    )
                    stats.update(
                    {
                        "actor/pg_loss": pg_loss.detach().item(),
                        "actor/pg_clipfrac": pg_clipfrac.detach().item(),
                        "actor/ppo_kl": ppo_kl.detach().item(),
                        "actor/pg_clipfrac_lower": pg_clipfrac_lower.detach().item(),
                    }
                    )
                policy_loss = pg_loss

            if calculate_entropy:
                entropy = output["entropy"][:, -response_length - 1 : -1].contiguous()
                if not forward_only:
                    entropy_loss = agg_loss(loss_mat=entropy, loss_mask=response_mask, loss_agg_mode=loss_agg_mode)
                    entropy_coeff = meta_info["entropy_coeff"]
                    policy_loss = pg_loss - entropy_coeff * entropy_loss
                else:
                    ret_entropy = entropy

            if forward_only:
                policy_loss = torch.tensor(1.0, device=device)
            else:
                if self.config.use_kl_loss:
                    ref_log_prob = data["ref_log_prob"]
                    # compute kl loss
                    kld = kl_penalty(logprob=log_prob, ref_logprob=ref_log_prob, kl_penalty=self.config.kl_loss_type)
                    kl_loss = agg_loss(loss_mat=kld, loss_mask=response_mask, loss_agg_mode=self.config.loss_agg_mode)

                    policy_loss = policy_loss + kl_loss * self.config.kl_loss_coef
                    metrics["actor/kl_loss"] = kl_loss.detach().item()
                    metrics["actor/kl_coef"] = self.config.kl_loss_coef
                


                # return loss and stats

            append_to_dict(metrics, stats)
            return policy_loss, [metrics, ret_entropy]

        def forward_step(batch_iter, model):
            batch = next(batch_iter)
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"].to(bool)
            position_ids = batch["position_ids"]

            multi_modal_inputs = {}
            if "multi_modal_inputs" in batch:
                for key in batch["multi_modal_inputs"][0].keys():
                    idxs = batch["multi_modal_inputs_idx"]
                    mmi = batch["multi_modal_inputs"]
                    multi_modal_inputs[key] = torch.cat(
                        [mmi[idx].get(key) for idx in idxs if mmi[idx].get(key) is not None], dim=0
                    )
            responses = batch["responses"]
            response_length = responses.size(1)
            label = position_ids.clone()
            label[:, -response_length - 1 : -1] = responses
            label_mask = attention_mask.clone()
            label_mask[:, : -response_length - 1] = False
            label_mask[:, -1] = False

            from verl.models.mcore import get_mcore_forward_fn, get_mcore_forward_fused_fn

            if self.use_fused_kernels:
                forward_fn = get_mcore_forward_fused_fn(self.hf_config)
                # return dict of [logits, entropy]
                output = forward_fn(
                    model,
                    input_ids,
                    position_ids,
                    attention_mask,
                    sequence_parallel=self.tf_config.sequence_parallel,
                    multi_modal_inputs=multi_modal_inputs,
                    labels=label,
                    labels_mask=label_mask,
                )
            else:
                forward_fn = get_mcore_forward_fn(self.hf_config)

                def logits_processor(logits, label, label_mask):
                    assert logits.shape[:2] == label.shape[:2]
                    assert label.shape == label_mask.shape
                    ret = {}
                    if calculate_entropy:
                        logits_bak = logits.clone()
                        logger.warning_once(
                            "For memory-efficient computation, enable fused kernels via "
                            "`actor_rollout_ref.model.use_fused_kernels=True`. "
                            "The current `clone()` operation ensures correctness but increases memory usage."
                        )
                        entropy = vocab_parallel_entropy(logits)
                        ret["entropy"] = entropy
                    else:
                        logits_bak = logits
                    log_probs = vocab_parallel_log_probs_from_logits(logits_bak, label)
                    log_probs = log_probs.masked_fill(~label_mask, 0.0)
                    ret["log_probs"] = log_probs
                    return ret

                logits_processor_args = {"label": label, "label_mask": label_mask}
                output = forward_fn(
                    model,
                    input_ids,
                    attention_mask,
                    position_ids,
                    sequence_parallel=self.tf_config.sequence_parallel,
                    multi_modal_inputs=multi_modal_inputs,
                    logits_processor=logits_processor,
                    logits_processor_args=logits_processor_args,
                )

            if forward_only:
                meta_info = None
            else:
                clip_ratio_c = self.config.get("clip_ratio_c", 3.0)
                meta_info = {
                    "clip_ratio": self.config.clip_ratio,
                    "entropy_coeff": self.config.entropy_coeff,
                    "clip_ratio_c": clip_ratio_c,
                }
            return output, partial(loss_func, data=batch, meta_info=meta_info)

        # batch should be a list of batches inside micro-batches
        batch_generator = make_batch_generator(micro_batches, vpp_size=len(self.actor_module))

        # TODO: we may use the new schedule instead
        # for flash-attn: (seq_len, batch_size, hidden_size) = (mbs*seq_len, 1, hidden_size)
        if mpu.get_pipeline_model_parallel_world_size() > 1:
            losses_reduced = forward_backward_func(
                forward_step_func=forward_step,
                data_iterator=batch_generator,
                model=self.actor_module,
                num_microbatches=n_micro_batch,
                seq_length=total_seqlen,  # no use when input_shapes was set
                micro_batch_size=1,  # no use when input_shapes was set
                forward_only=forward_only,
            )
        else:
            losses_reduced = forward_backward_func(
                forward_step_func=forward_step,
                data_iterator=batch_generator,
                model=self.actor_module,
                num_microbatches=n_micro_batch,
                seq_length=total_seqlen,  # in use for pp = 1
                micro_batch_size=1,  # in use for pp = 1
                forward_only=forward_only,
            )
        # loss_reduces contains the stats returned from loss_func
        '''
        loss_reduces包含从loss_func中得到的stats
        '''

        if self.has_multi_modal_inputs:
            data.batch.pop("multi_modal_inputs")
            data.batch.pop("multi_modal_inputs_idx")
            data.non_tensor_batch.pop("multi_modal_inputs")

        losses_reduced = {"output": losses_reduced}
        if use_dynamic_bsz:
            losses_reduced["indices"] = indices
        return losses_reduced


- 这段代码定义了一个名为 forward_backward_batch 的核心函数，它是大模型强化学习（特别是 PPO 算法）中用于执行前向传播（计算模型输出）和反向传播（计算梯度）的关键组件。
- <font color='red'>这段代码通常运行在分布式训练框架（如 Megatron-LM）之上，负责将一个大的训练批次（Batch）切分成微批次（Micro-batch），并行处理以节省显存。</font>

以下是对该函数的详细拆解与解释：

## 1. 函数定义与参数解析
- 核心功能：<font color='red'>处理输入数据，执行模型的前向和反向计算。</font>
- 关键参数：
    -<font color='red'> data: 输入数据，包含：</font>
        - 提示词（prompts）、
        - 参考模型的对数概率（old_log_probs）、
        - 优势函数（advantages）等 PPO 训练所需信息。
    - forward_only: 布尔值。如果为 True，只做前向传播（用于推理或评估）；如果为 False，则进行完整的训练（包含反向传播）。
    - use_dynamic_bsz: 是否使用动态批次切分。如果开启，切分逻辑会根据 Token 总数而非固定样本数来切分，这对处理变长序列更高效。
    - micro_batch_size: 微批次大小。如果未开启动态切分，就用这个固定数值切分数据。
    
## 2. 数据预处理与切分 (Data Preparation)
这部分代码负责将一个大批次的数据整理好，并切分成小块，以便流水线并行处理。
- 跨 Rank 通信：
    - <font color='red'> broadcast_dict_tensor(...): 在流水线并行（Pipeline Parallelism, PP）中，数据通常只在最后一个 PP Rank 上，这里将其广播到所有其他 Rank，确保所有 GPU 都有相同的数据。</font>
    
- 多模态处理：
    - 代码检查了 multi_modal_inputs（如图片等），<font color='red'>如果存在则进行特殊的拼接处理，这是为了适配 Qwen2-VL 等多模态模型的结构。</font>
    
- 动态切分逻辑：
    - 如果 use_dynamic_bsz=True，调用 rearrange_micro_batches。它会根据 max_token_len（最大 Token 长度）来智能切分，防止显存溢出（OOM）。
    - 如果是固定切分，则直接按 micro_batch_size 切分。
    
## 3. 损失函数定义 (Loss Function)
这是 PPO 算法的核心数学逻辑，定义在 loss_func 内部函数中。
- 推理模式：如果 forward_only=True，仅执行后处理函数（post_process_fn）收集指标，不计算梯度。
- 训练模式：
    - 策略梯度损失 (Policy Loss)：计算新旧策略之间的差异。
        - <font color='red'>支持多种 Loss 模式（loss_mode），代码中特别展示了 future_kl 模式，该模式会计算大量详细的统计信息（如 influence_weights, clipfrac 等）。</font>
        - 计算 pg_loss（策略梯度损失）、pg_clipfrac（被裁剪的比例）和 ppo_kl（KL 散度）。
    - 熵正则化 (Entropy Regularization)：如果 calculate_entropy=True，计算熵损失并从总损失中减去（最大化熵以鼓励探索）。
    - KL 散度损失 (KL Loss)：如果配置中启用了 use_kl_loss，计算当前策略与参考策略（Reference Model）之间的 KL 散度，并将其作为惩罚项加入总损失。
- 指标收集：函数返回总损失（policy_loss）以及包含各种调试和监控指标的字典（metrics）
    
## 4. 前向步骤 (Forward Step)
定义了如何将数据送入模型。
- 输入构建：
    - 从 Batch 中提取 input_ids, attention_mask, position_ids。
    - 构建 labels（标签）：代码巧妙地将 label 的非回答部分掩码掉，只计算模型生成“回答”部分的 Loss。
- 混合精度与内核优化：
    - use_fused_kernels: 如果启用，使用融合内核（Fused Kernels）来同时计算 Logits 和 Entropy，这能显著节省显存（避免中间变量的存储）。
    - 如果不启用融合内核，代码会发出警告，提示显存效率较低。
    
## 5. 执行与返回
- 调度执行：根据流水线并行的world大小，调用 forward_backward_func（通常来自 DeepSpeed 或 Megatron 的调度器）。该函数负责管理微批次的流水线气泡，执行前向和反向传播。
- 结果清理与返回：收集所有 Reduce 后的损失和指标。如果使用了动态批处理，还会返回索引信息
    
## 5. 总结与核心逻辑图解
这段代码的本质是一个PPO 训练循环的微观执行单元。它的执行流程如下：
1. 输入：接收一个包含 Prompts 和 Critic 生成的 Advantages 的大批次数据。
2. 切分：为了适应显存，将大批次切成一个个 Micro-batch。
3. 计算：
    - Forward: 模型生成回答，计算 Logits 和 Log_probs。
    - Loss: 结合旧策略的 Log_probs 和 Advantages，计算 PPO 损失。
4. 输出：返回损失值和统计指标（如 KL 散度、梯度范数等），供优化器（如 Adam）更新模型参数。

- 关键点： 代码中大量关于 sequence_parallel、virtual_pipeline_model_parallel 的处理，表明这是为超大规模模型（如 Qwen2-72B 或更大）设计的分布式训练代码，旨在解决单卡显存不足的问题。

# 闭包

- 将 loss_func 和 forward_step 定义在 forward_backward_batch 函数内部，是深度学习框架（特别是 Megatron-LM、DeepSpeed 等大规模训练框架）中一种非常经典且必要的高阶函数（Higher-Order Function）设计模式。

- 这种做法的核心原因是为了解决作用域隔离、数据闭包传递以及框架接口标准化的问题。

结合你上传的代码，以下是详细的解释：
## 1. 闭包（Closure）：传递“当前批次”的特定数据
这是最直接的原因。PPO（近端策略优化）算法的 Loss 计算非常复杂，它不仅需要模型的输出（Logits），还需要当前批次（Batch）中存储的大量“旧”数据来进行对比。
- 需要访问的数据：
    - data["old_log_probs"]: 旧策略下的对数概率。
    - data["advantages"]: 优势函数估计值。
    - data["responses"]: 实际生成的回复（用于计算 Mask）。
    - 配置参数：如 self.config.entropy_coeff（熵系数）、self.config.kl_loss_coef（KL散度系数）。
- <font color='red'>如果这两个函数定义在外部，它们将无法直接访问 forward_backward_batch 内部的局部变量（如 data, config）。虽然可以通过参数列表显式传递，但这会破坏框架的标准接口。</font>

- 通过在内部定义，这两个函数自动形成了一个闭包，可以直接“捕获”外部函数作用域内的这些变量。你看代码中的 loss_func，它直接使用了 data 和 meta_info，而不需要将它们作为参数传入。

## 2. 框架接口的标准化（Standardization）
在 Megatron-LM 或 DeepSpeed 的 forward_backward_func（即代码末尾调用的那个调度器）中，框架期望接收的函数签名是固定的。
- 框架期望的 forward_step 签名：通常只接受 (model, batch) 两个参数。
- 实际情况：你的模型前向传播可能需要 input_ids, mask, position_ids, 甚至 multi_modal_inputs。

### 解决方案：
- 在外部函数中将这些参数打包或处理好，然后在内部定义的 forward_step 中进行解包和逻辑处理。这样，对外暴露给调度器的 forward_step 依然符合 (model, batch) 的标准，而内部逻辑可以非常灵活地处理复杂的业务需求（如 Qwen2-VL 的多模态处理逻辑）。

## 3. 状态隔离（State Isolation）

forward_backward_batch 是按批次（Batch）执行的。每次调用这个函数，处理的都是全新的数据。
- 动态性：每个 Batch 的 data 是不同的，每个 Batch 的 Loss 计算逻辑（虽然代码一样，但依赖的数据不同）也是针对当前 Batch 的。
- 避免全局状态污染：如果将 loss_func 定义为全局函数，它可能会依赖全局变量，导致在多线程或多 GPU 环境下出现数据竞争或状态混乱。
- <font color='red'>每次调用 forward_backward_batch 时，loss_func 都绑定到当前批次的特定数据，执行完即销毁，彻底避免状态残留</font>

将它们定义在内部，确保了每次执行 forward_backward_batch 时，都会创建一组全新的、独立的函数实例，它们只与当前这一次调用的上下文绑定，执行完即销毁，非常安全。

## 4. 逻辑聚合（Logical Cohesion）
从代码可读性和维护性的角度看，将这三个部分放在一起非常合理：
1. 主流程 (forward_backward_batch)：负责数据切分、并行通信。
2. 前向逻辑 (forward_step)：负责如何喂数据给模型。
3. 损失逻辑 (loss_func)：负责如何根据模型输出和参考数据算 Loss。

这三者是紧密耦合的。如果你把 loss_func 搬到另一个文件，当你要修改 PPO 的奖励计算逻辑时，就需要在两个地方跳来跳去。把它们放在一起，符合“高内聚”的设计原则。

## 总结

这种设计模式可以理解为“工厂模式”或“闭包封装”：
1. forward_backward_batch 是一个工厂：它接收原材料（Data）和配方（Config）。
2. 它制造出两个工具：forward_step（用来加工模型）和 loss_func（用来质检）。
3. 它将工具连同当前批次的原料一起交给流水线调度器（forward_backward_func）去执行。

如果不这样做，你就需要修改底层框架（Megatron）的源码来适配你复杂的 PPO Loss 计算逻辑，这将导致代码难以维护且无法复用。

# get_forward_backward_func :Megatron-LM 的并行调度核心

- get_forward_backward_func() 是 Megatron-LM 框架中<font color='red'>动态选择前向-反向传播调度策略的核心工厂函数。</font>

- 它根据当前分布式训练的<font color='red'>并行配置</font>（特别是 pipeline 模型并行的设置），<font color='red'>返回一个能正确协调前向/反向传播、设备通信和梯度计算的调度函数。</font>
- 其设计体现了 "<font color='red'>配置驱动执行流</font>" 的分布式训练哲学。

###### 调度器的智能路由: 
- 非最后一个 stage：跳过损失计算，仅传递激活值
- <font color='red'>最后一个 stage：必须调用 loss_func 生成标量损失用于反向传播</font>

## 一、核心目标与设计思想
### 1. 核心目标
- 动态适配并行策略： <font color='red'>根据 parallel_state（全局并行状态）自动选择最优的前向-反向传播实现。</font>
- <font color='red'>解耦计算逻辑与调度逻辑</font>：将模型计算（forward_step_func）与分布式调度分离，提升框架灵活性。
- 统一接口，隐藏复杂性：<font color='red'>对上层训练循环暴露一致的调用接口，屏蔽底层并行细节。</font>

### 2. 关键设计思想
| 设计原则                | 实现方式                                                                 |
|-------------------------|--------------------------------------------------------------------------|
| 条件化调度          | 通过 `pipeline_model_parallel_size` 动态选择 3 种调度器                   |
| 零侵入式扩展        | <font color='red'>新增并行策略只需实现新调度函数，无需修改上层训练逻辑 </font>                    |
| 数据驱动形状管理    | 通过 `adjust_tensor_shapes_fn` 动态处理序列并行等场景的张量形状变化      |

## 决策树可视化

In [ ]:
pipeline_model_parallel_size?
│
├── 1 → forward_backward_no_pipelining
│
└── >1 → virtual_pipeline_model_parallel_world_size?
     │
     ├── 存在 → forward_backward_pipelining_with_interleaving (Interleaved)
     │
     └── 不存在 → forward_backward_pipelining_without_interleaving (传统 1F1B)

## 关键决策依据

| 决策条件                                      | 对应调度器                                      | 适用场景                                                                 |
|-----------------------------------------------|------------------------------------------------|--------------------------------------------------------------------------|
| `pipeline_model_parallel_size == 1`           | `forward_backward_no_pipelining`               | 单设备训练 / 仅使用张量并行（Tensor Parallelism）                        |
| `pipeline_model_parallel_size > 1` 且 无虚拟 pipeline | `forward_backward_pipelining_without_interleaving` | 传统 pipeline 并行（<font color='red'>每个设备只负责 1 个 stage</font> ）                        |
| `pipeline_model_parallel_size > 1` 且 <font color='red'>有虚拟 pipeline </font>| `forward_backward_pipelining_with_interleaving`  | Interleaved pipeline 并行（<font color='red'>单设备负责多个 stage，减少 pipeline 气泡</font>） |

虚拟 pipeline (Interleaved)：允许单个物理设备在 pipeline 中承担 多个逻辑 stage（例如 4-stage pipeline 中，2 个设备各负责 stage1+3 和 stage2+4），显著提升设备利用率。

## 调度器必须完成的核心任务

| 任务类型                | 实现细节                                                                 |
|-------------------------|--------------------------------------------------------------------------|
| 前向传播调度        | 按 pipeline 阶段拆分 microbatches，协调设备间激活值传递                  |
| 损失计算触发        | <font color='red'>在最后一个 pipeline stage 调用 `loss_func`（见下文重点说明）</font>         |
| 反向传播调度        | <font color='red'>按 pipeline 阶段反向协调梯度计算，处理设备间梯度传递</font>                   |
| 梯度累积管理        | 对 `num_microbatches` 个 microbatch 的梯度求和后再更新                   |
| 通信优化            | <font color='red'>重叠计算与通信（如 `torch.distributed.all_reduce`） </font>                     |
| 序列并行适配        | 通过 `adjust_tensor_shapes_fn` 动态调整张量形状（如 `seq_len /= tp_size`） |



## <font color='red'>Interleaved （交替）</font>模式的革命性改进
- 传统 pipeline 的 pipeline bubble 问题：
    - 设备空闲率高达 50%（以 4-stage 为例）

In [ ]:
设备0: [S1][S1][S1][S1][S1][S1][S1][S1]  (S1=stage1)
设备1:    [S2][S2][S2][S2][S2][S2][S2]     (S2=stage2)
设备2:       [S3][S3][S3][S3][S3][S3]      (S3=stage3)
设备3:          [S4][S4][S4][S4][S4]       (S4=stage4)

- Interleaved 模式的解决方案（2 设备运行 4-stage pipeline）：
    - 设备利用率提升至 100%（无气泡）
    - get_forward_backward_func() 通过 virtual_pp_size 检测此配置，返回 with_interleaving 调度器

In [ ]:
设备0: [S1][S3][S1][S3][S1][S3][S1][S3]  # 同时负责 stage1 和 stage3
设备1:    [S2][S4][S2][S4][S2][S4][S2][S4] # 同时负责 stage2 和 stage4

## 为什么需要此设计？—— 分布式训练的核心挑战

| 挑战                          | 本设计的解决方案                                                                 |
|-------------------------------|-------------------------------------------------------------------------------|
| 并行策略碎片化            | 用单一接口适配 3 种并行模式，避免训练循环条件分支爆炸                          |
| 损失计算依赖上下文        | 通过 `forward_step_func` 返回的闭包传递 microbatch 特定数据（如 `loss_mask`） |
| pipeline 气泡浪费算力     | Interleaved 模式通过虚拟 pipeline 消除气泡                                   |
| 序列并行的动态形状变化    | `adjust_tensor_shapes_fn` 集中处理张量形状适配                                |

## 总结：分布式训练的"交通指挥中心"

get_forward_backward_func() 本质是 Megatron-LM 的分布式执行引擎调度器：

1. 智能决策：根据 parallel_state 自动选择最优调度策略（3 种模式）。
2. 精准控制：<font color='red'>仅在 pipeline 末端触发 loss_func，避免无效计算。</font>
3. <font color='red'>上下文传递：利用 闭包机制 将 microbatch 数据安全传递给损失函数。</font>
4. <font color='red'>硬件优化：通过 Interleaved 模式消除 pipeline 气泡，序列并行适配提升扩展性。</font>

- <font color='red'>理解此函数是掌握 Megatron-LM 分布式训练逻辑的关键</font> ——<font color='green'>它将复杂的并行通信、计算调度、上下文管理封装在统一接口下</font>，使用户只需关注 forward_step_func 和 loss_func 的实现，而无需处理底层分布式细节。

- 这种 "策略选择 + 接口统一" 的设计，是工业级分布式训练框架的核心范式。

In [ ]:
def get_forward_backward_func():
    """Retrieves the appropriate forward_backward function given the
    configuration of parallel_state.

    Returns a function that will perform all of the forward and
    backward passes of the model given the pipeline model parallel
    world size and virtual pipeline model parallel world size in the
    global parallel_state.

    Note that if using sequence parallelism, the sequence length component of
    the tensor shape is updated to original_sequence_length /
    tensor_model_parallel_world_size.

    The function returned takes the following arguments:

    forward_step_func (required): A function that takes a data
        iterator and a model as its arguments and return the model's
        forward output and the loss function. The loss function should
        take one torch.Tensor and return a torch.Tensor of loss and a
        dictionary of string -> torch.Tensor.

        A third argument, checkpoint_activations_microbatch, indicates
        that the activations for this microbatch should be
        checkpointed. A None value for this argument indicates that
        the default from the configuration should be used. This is
        used when the
        num_microbatches_with_partial_activation_checkpoints is used.

        For example:

        def loss_func(loss_mask, output_tensor):
            losses = output_tensor.float()
            loss_mask = loss_mask.view(-1).float()
            loss = torch.sum(losses.view(-1) * loss_mask) / loss_mask.sum()

            # Reduce loss for logging.
            averaged_loss = average_losses_across_data_parallel_group([loss])

            return loss, {'lm loss': averaged_loss[0]}

        def forward_step(data_iterator, model):
            data, loss_mask = next(data_iterator)
            output = model(data)
            return output, partial(loss_func, loss_mask)


        forward_backward_func(forward_step_func=forward_step, ...)


    data_iterator (required): an iterator over the data, will be
        passed as is to forward_step_func. Expected to be a list of
        iterators in the case of interleaved pipeline parallelism.

    model (required): the actual model. Expected to be a list of modules in the case of interleaved
        pipeline parallelism. Must be a (potentially wrapped) megatron.core.models.MegatronModule.

    num_microbatches (int, required):
        The number of microbatches to go through

    seq_length (int, required): Sequence length of the current global batch. If this is a dual-stack
        transformer, this is the encoder's sequence length. This is ignored if variable_seq_lengths
        in the config is True. Otherwise, each microbatch in the current global batch size must use
        this sequence length.

    micro_batch_size (int, required): The number of sequences in a microbatch.

    decoder_seq_length (int, optional): The sequence length for the decoder in a dual-stack
        transformer. This is ignored for a single-stack transformer.

    forward_only (optional, default = False): Perform only the forward step

    collect_non_loss_data (optional, bool, default=False): TODO

    first_val_step (bool, optional): Is the first step of the validation phase. Used by
        Transformer Engine modules to only update their fp8 weights only on the first validation
        step.

    adjust_tensor_shapes_fn (Callable, optional): A function that adjusts the receive and send
        tensor shapes. Only applicable in forward_backward_pipelining_without_interleaving for now.
        Takes in a list of receive shapes and a list of send shapes and returns the adjusted
        respective list of shapes. Thus it is not used in the other forward-backward functions
        which have different shape handling.

    """
    pipeline_model_parallel_size = parallel_state.get_pipeline_model_parallel_world_size()
    
    if pipeline_model_parallel_size > 1:
        '''
        情况2: 有 pipeline 并行:
        
        '''
        # 子情况2.1: 使用虚拟 pipeline (Interleaved 模式)
        if parallel_state.get_virtual_pipeline_model_parallel_world_size() is not None:
            forward_backward_func = forward_backward_pipelining_with_interleaving
        
        # 子情况2.2: 传统 pipeline (无虚拟 pipeline)
        else:
            forward_backward_func = forward_backward_pipelining_without_interleaving
    else:
        '''
        无 pipeline 并行 (单设备/仅张量并行)
        '''
        forward_backward_func = forward_backward_no_pipelining
        
    return forward_backward_func

方法在megatron/core/pilelin_parallel/schedules.py https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py

- forward_backward_pipelining_with_interleaving https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L889
- forward_backward_pipelining_without_interleaving https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L2026
- forward_backward_no_pipelining https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L591

